In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from keras.models import Model
from keras.layers import Input, Conv1D, MaxPooling1D, Dense, Dropout, Flatten, Concatenate, LSTM
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
import os
import random

In [3]:
back_data= pd.read_csv('back_data_log_transformed_price.csv')
lay_data= pd.read_csv('lay_data_log_transformed_price.csv')
actual_price= pd.read_csv('actual_price.csv')

In [4]:
SEED = 0
def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    np.random.seed(seed)
    
def set_global_determinism(seed=SEED):
    set_seeds(seed=seed)

    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
    
def model_eval(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)*100
    rmse = np.sqrt(mse)
    
    print(f"\nModel Performance:")
    print(f"MSE: {mse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MAPE: {mape:.4f} %")
    print(f"RMSE: {rmse:.4f}")

In [45]:
class CNNLSTM_Forecaster:
    def __init__(self, sequence_length=10, forecast_horizon=1):
        self.sequence_length = sequence_length
        self.forecast_horizon = forecast_horizon
        self.model = None
        self.time_scaler = None
        self.external_scaler = None
        
    def prepare_data(self, df, train_idx, test_idx, target_col_index=-1):
        
        df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]
        
        external_factors_train = df_train.iloc[:, :2].values
        time_series_data_train = df_train.iloc[:, 2:].values
        external_factors_test = df_test.iloc[:, :2].values
        time_series_data_test = df_test.iloc[:, 2:].values
        
        # Scale the data
        #self.time_scaler = MinMaxScaler()
        #self.external_scaler = StandardScaler()
        
        #time_series_scaled_train = self.time_scaler.fit_transform(time_series_data_train)
        #external_factors_scaled_train = self.external_scaler.fit_transform(external_factors_train)
        
        #time_series_scaled_test = self.time_scaler.transform(time_series_data_test)
        #external_factors_scaled_test = self.external_scaler.transform(external_factors_test)

        def process(time_series_scaled,external_factors_scaled):
            # Create sequences
            X_time_sequences = []
            X_external_features = []
            y_targets = []
            
            for i in range(len(time_series_scaled)): 
                time_series = time_series_scaled[i] 
                external = external_factors_scaled[i]
                
                for start_idx in range(len(time_series) - self.sequence_length - self.forecast_horizon + 1):
                    # Input sequence
                    sequence = time_series[start_idx:start_idx + self.sequence_length]
                    X_time_sequences.append(sequence)
                    
                    X_external_features.append(external)
                    
                    # Target (next time point(s))
                    if self.forecast_horizon == 1:
                        target = time_series[start_idx + self.sequence_length]
                    else:
                        target = time_series[start_idx + self.sequence_length:start_idx + self.sequence_length + self.forecast_horizon]
                    y_targets.append(target)
            
            # Convert to numpy arrays and reshape
            X_time = np.array(X_time_sequences).reshape(-1, self.sequence_length, 1)
            X_external = np.array(X_external_features)
            y = np.array(y_targets)
            
            return X_time, X_external, y

        #X_time_train, X_external_train, y_train = process(time_series_scaled_train, external_factors_scaled_train)
        #X_time_test, X_external_test, y_test = process(time_series_scaled_test, external_factors_scaled_test)
        
        X_time_train, X_external_train, y_train = process(time_series_data_train, external_factors_train)
        X_time_test, X_external_test, y_test = process(time_series_data_test, external_factors_test)
        
        return X_time_train, X_external_train, y_train, X_time_test, X_external_test, y_test
    
    def build_model(self, external_features=2):
        
        # Time series input (CNN-LSTM branch)
        time_input = Input(shape=(self.sequence_length, 1), name='time_input')
        
        # CNN layers for pattern recognition in time series
        conv1 = Conv1D(filters=32, kernel_size=2, activation='relu', padding='same')(time_input)
        conv1 = MaxPooling1D(pool_size=2)(conv1)
        conv1 = Dropout(0.2)(conv1)
        
        conv2 = Conv1D(filters=16, kernel_size=2, activation='relu', padding='same')(conv1)
        conv2 = Dropout(0.2)(conv2)
        
        # LSTM layers for sequence modeling
        lstm1 = LSTM(32, return_sequences=True, dropout=0.2)(conv2)
        lstm2 = LSTM(16, return_sequences=False, dropout=0.2)(lstm1)
        
        # External factors input
        external_input = Input(shape=(external_features,), name='external_input')
        external_dense = Dense(16, activation='relu')(external_input)
        external_dense = Dropout(0.2)(external_dense)
        
        # Combine both branches
        combined = Concatenate()([lstm2, external_dense])
        
        # Final prediction layers
        dense1 = Dense(32, activation='relu')(combined)
        dense1 = Dropout(0.3)(dense1)
        dense2 = Dense(16, activation='relu')(dense1)
        
        # Output layer
        if self.forecast_horizon == 1:
            output = Dense(1, activation='linear', name='output')(dense2)
        else:
            output = Dense(self.forecast_horizon, activation='linear', name='output')(dense2)
        
        # Create model
        model = Model(inputs=[time_input, external_input], outputs=output)
        
        # Compile model
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss='mse',
            metrics=['mae']
        )
        
        self.model = model
        return model
    
    def train(self, X_time, X_external, y, validation_split=0.2, epochs=100, batch_size=32):
        
        # Callbacks
        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True,
            verbose=0
        )
        
        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=10,
            min_lr=0.0001,
            verbose=0
        )

        
        # Train the model
        history = self.model.fit(
            [X_time, X_external], y,
            batch_size=batch_size,
            epochs=epochs,
            validation_split=validation_split,
            callbacks=[early_stopping, reduce_lr],
            verbose=0
        )
        
        return history
    
    def predict(self, X_time, X_external):
        predictions = self.model.predict([X_time, X_external])
        return predictions
    
    def evaluate_model(self, y_true, y_pred):
        """
        Evaluate model performance
        """
        mse = mean_squared_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        mape = mean_absolute_percentage_error(y_true, y_pred)*100
        rmse = np.sqrt(mse)
        
        print(f"\nModel Performance:")
        print(f"MSE: {mse:.4f}")
        print(f"MAE: {mae:.4f}")
        print(f"MAPE: {mape:.4f} %")
        print(f"RMSE: {rmse:.4f}")
        
        return {'mse': mse, 'mae': mae, 'mape': mape, 'rmse': rmse}

In [46]:
def forecasts(df, train_idx, test_idx):
    forecaster = CNNLSTM_Forecaster()
    
    X_time_train, X_external_train, y_train, X_time_test, X_external_test, y_test = forecaster.prepare_data(df, train_idx, test_idx)

    print(X_time_train.shape, X_external_train.shape, y_train.shape, X_time_test.shape, X_external_test.shape, y_test.shape)

    model = forecaster.build_model(external_features=2)
    #model.summary()
    
    history = forecaster.train(
        X_time_train, X_external_train, y_train,
        validation_split=0.2,
        epochs=50,
        batch_size=64
    )
    
    # Predict and evaluate
    y_pred = forecaster.predict(X_time_test, X_external_test)
    metrics = forecaster.evaluate_model(y_test, y_pred.flatten())

    return {'forecast':y_pred, 'model':forecaster}

In [47]:
commisions = 0.05

def riskless_profit(cur_back, cur_lay, future_back, future_lay, forecast_back, forecast_lay):
    p = np.nan

    forecast_p1 = (forecast_back - cur_lay + commisions * (1 - forecast_back)) / \
                  (forecast_back * cur_lay - forecast_back + cur_lay - commisions)
    forecast_p2 = (cur_back - forecast_lay + commisions * (1 - cur_back)) / \
                  (cur_back * forecast_lay - cur_back + forecast_lay - commisions)

    if forecast_p1 > forecast_p2 and forecast_p1 > 0:
        p = (future_back - cur_lay + commisions * (1 - future_back)) / \
            (future_back * cur_lay - future_back + cur_lay - commisions)
    elif forecast_p2 > forecast_p1 and forecast_p2 > 0:
        p = (cur_back - future_lay + commisions * (1 - cur_back)) / \
            (cur_back * future_lay - cur_back + future_lay - commisions)

    return p

actual_price = pd.read_csv('actual_price.csv').iloc[test_idx]
cur_back, cur_lay = np.array( actual_price['cur_back'] ), np.array( actual_price['cur_lay'] )
future_back, future_lay = np.array( actual_price['future_back'] ), np.array( actual_price['future_lay'] )

perfect_riskless_profit = np.array([
    riskless_profit(cb, cl, fb, fl, fob, fol)
    for cb, cl, fb, fl, fob, fol in zip(
        cur_back, cur_lay,
        future_back, future_lay,
        future_back, future_lay
    )
])
    
def profits_summary(x, perfect_riskless_profit=perfect_riskless_profit):
    x = np.array(x)
    s = {}

    x_valid = x[~np.isnan(x)]
    prp_valid = perfect_riskless_profit[~np.isnan(perfect_riskless_profit)]

    s["Min"] = np.min(x_valid) if len(x_valid) > 0 else np.nan
    s["1st Qu."] = np.percentile(x_valid, 25) if len(x_valid) > 0 else np.nan
    s["Median"] = np.median(x_valid) if len(x_valid) > 0 else np.nan
    s["Mean"] = np.mean(x_valid) if len(x_valid) > 0 else np.nan
    s["3rd Qu."] = np.percentile(x_valid, 75) if len(x_valid) > 0 else np.nan
    s["Max"] = np.max(x_valid) if len(x_valid) > 0 else np.nan
    s["Betted Number"] = len(x_valid)
    s["Betting Rate in Total Race"] = len(x_valid) / len(x)
    s["Betting Rate in Profitable Race"] = len(x_valid) / len(prp_valid) if len(prp_valid) > 0 else np.nan
    s["Profit Sum"] = np.sum(x_valid)
    s["Profit Sum/Total Possible Profit"] = np.sum(x_valid) / np.sum(prp_valid) if np.sum(prp_valid) != 0 else np.nan
    s["Positive Rate in Betted Race"] = np.sum(x_valid > 0) / len(x_valid) if len(x_valid) > 0 else np.nan

    return pd.Series(s)

In [48]:
set_global_determinism(seed=SEED)

indices = np.arange(len(back_data))
train_idx, test_idx = train_test_split(indices, test_size=0.5, random_state=42)

back_CNNLSTM_results = forecasts(back_data.iloc[:,2:], train_idx, test_idx)
lay_CNNLSTM_results = forecasts(lay_data.iloc[:,2:], train_idx, test_idx)

(7215, 10, 1) (7215, 2) (7215,) (7216, 10, 1) (7216, 2) (7216,)


/opt/anaconda3/envs/ML/lib/python3.11/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['time_input', 'external_input']. Received: the structure of inputs=('*', '*')
  warnings.warn(


226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

Model Performance:
MSE: 0.0991
MAE: 0.2346
MAPE: 11.9488 %
RMSE: 0.3148
(7215, 10, 1) (7215, 2) (7215,) (7216, 10, 1) (7216, 2) (7216,)


/opt/anaconda3/envs/ML/lib/python3.11/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['time_input', 'external_input']. Received: the structure of inputs=('*', '*')
  warnings.warn(


226/226 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Model Performance:
MSE: 0.2064
MAE: 0.3183
MAPE: 13.7941 %
RMSE: 0.4543


In [49]:
profits_summary(perfect_riskless_profit)

Min                                    0.000001
1st Qu.                                0.001049
Median                                 0.002990
Mean                                   0.006174
3rd Qu.                                0.007468
Max                                    0.229826
Betted Number                       1915.000000
Betting Rate in Total Race             0.265382
Betting Rate in Profitable Race        1.000000
Profit Sum                            11.823833
Profit Sum/Total Possible Profit       1.000000
Positive Rate in Betted Race           1.000000
dtype: float64

In [54]:
#back_time_scaler = back_CNNLSTM_results['model'].time_scaler
#lay_time_scaler = lay_CNNLSTM_results['model'].time_scaler
#forecast_back = ( back_CNNLSTM_results['forecast'].ravel() - back_time_scaler.min_[-1] ) / back_time_scaler.scale_[-1]
#forecast_lay = ( lay_CNNLSTM_results['forecast'].ravel() - lay_time_scaler.min_[-1] ) / lay_time_scaler.scale_[-1]
forecast_back = np.exp(back_CNNLSTM_results['forecast'].ravel())
forecast_lay = np.exp(lay_CNNLSTM_results['forecast'].ravel())

In [55]:
CNNLSTM_riskless_profit = np.array([
    riskless_profit(cb, cl, fb, fl, fob, fol)
    for cb, cl, fb, fl, fob, fol in zip(
        cur_back, cur_lay,
        future_back, future_lay,
        forecast_back, forecast_lay
    )
])
profits_summary(CNNLSTM_riskless_profit)

Min                                   -0.091828
1st Qu.                               -0.014951
Median                                -0.006653
Mean                                  -0.009393
3rd Qu.                               -0.001838
Max                                    0.229826
Betted Number                       5821.000000
Betting Rate in Total Race             0.806680
Betting Rate in Profitable Race        3.039687
Profit Sum                           -54.675460
Profit Sum/Total Possible Profit      -4.624174
Positive Rate in Betted Race           0.118365
dtype: float64

In [56]:
model_eval(future_back,forecast_back)


Model Performance:
MSE: 3011.9118
MAE: 15.8001
MAPE: 22.7269 %
RMSE: 54.8809


In [57]:
model_eval(future_lay,forecast_lay)


Model Performance:
MSE: 6749.4703
MAE: 24.4898
MAPE: 26.3726 %
RMSE: 82.1552


2. Model Architecture

CNN Branch: Extracts local patterns from time series using Conv1D layers

LSTM Branch: Captures long-term dependencies in the sequence

External Factors Branch: Processes external factors through dense layers

Combined Architecture: Concatenates CNN-LSTM features with external factors

3. Training Features

Early stopping to prevent overfitting
Learning rate reduction on plateau
Validation split for monitoring performance
